# Проект модуля. Нейросеть для автодополнения текстов

## Бизнес-контекст и задача проекта
Соцсетевое приложение, где пользователи постят короткие тексты. В продукте стоит задача — добавить возможность автодополнения текстов. Необходимо оздать модель, которую можно запускать на мобильных устройствах. Для смартфонов есть значительные требования по оперативной памяти и скорости работы, так что важна легковесность модели. 

## Формальная задача
Разработчики предоставили вам датасет с короткими текстовыми постами. 
### Поэтапное описание задачи:
- Взять датасет от разработчиков, очистить его, подготовить для обучения модели.
- Реализовать и обучить модель на основе рекуррентных нейронных сетей.
- Замерить качество разработанной и обученной модели.
- Взять более «тяжёлую» предобученную модель из Transformers и замерить её качество.
- Проанализировать результаты и дать рекомендации разработчикам: стоит ли использовать лёгкую модель или лучше постараться поработать с ограничениями по памяти и использовать большую предобученную.

## Загрузка библиотек

In [1]:
#Автоматическая перезагрузка при изменениях
%load_ext autoreload
%autoreload 2

In [2]:
# Для MPS рекомендуется включить fallback на CPU для неподдерживаемых операций
import os

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

In [ ]:
from src.data_utils import load_and_clear_data, load_or_tokenize, create_sequences, create_data_split, build_vocab
from src.constants import BATCH_SIZE
from src.next_token_dataset import NextTokenDataset, collate_fn
from src.lstm_model import LSTMLanguageModel
from src.lstm_train import train_model
from src.lstm_test import test_model
from src.lstm_utils import save_model_weight
from src.eval_transformer_pipeline import evaluate_transformer
from src.constants import PAD_TOKEN

import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim

/root/projects/sprint_2_rnn/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Этап 1. Сбор и подготовка данных

In [ ]:
texts = load_and_clear_data()

✅ Загружено 1600000 текстов, после очистки осталось 1597104 текстов.
Примеры очищенных текстов: ["a that's a bummer. you shoulda got david carr of third day to do it. d", "is upset that he can't update his facebook by texting it... and might cry as a result school today also. blah!", 'i dived many times for the ball. managed to save 50 the rest go out of bounds', 'my whole body feels itchy and like its on fire', "no, it's not behaving at all. i'm mad. why am i here? because i can't see you all over there."]


In [ ]:
tokenized_texts = load_or_tokenize(texts)

Загрузка токенизированных данных из data/tokenized_texts.pkl...


In [ ]:
X, Y = create_sequences(tokenized_texts)

In [ ]:
display(X[0:1])
display(Y[0:1])

[['<BOS>',
  'a',
  'that',
  "'s",
  'a',
  'bummer',
  '.',
  'you',
  'shoulda',
  'got',
  'david',
  'carr',
  'of',
  'third',
  'day',
  'to',
  'do',
  'it',
  '.',
  'd']]

[['a',
  'that',
  "'s",
  'a',
  'bummer',
  '.',
  'you',
  'shoulda',
  'got',
  'david',
  'carr',
  'of',
  'third',
  'day',
  'to',
  'do',
  'it',
  '.',
  'd',
  '<EOS>']]

In [ ]:
X_train, Y_train, X_val, Y_val, X_test, Y_test = create_data_split(X, Y) 

Train: 1277683, Val: 159710, Test: 159711


In [ ]:
# Создание словаря
word2idx, idx2word = build_vocab(tokenized_texts)

vocab_size = len(word2idx)

Всего уникальных токенов: 369896
Топ-10 частых: [('<BOS>', 1597104), ('<EOS>', 1597104), ('i', 949180), ('!', 916243), ('.', 797560), ('to', 564942), ('the', 521098), (',', 483127), ('a', 383113), ('my', 315823)]
✅ Размер словаря: 19998 (ограничено с 369896)


In [ ]:
# Создание датасетов
train_dataset = NextTokenDataset(X_train, Y_train, word2idx)
val_dataset = NextTokenDataset(X_val, Y_val, word2idx)
test_dataset = NextTokenDataset(X_test, Y_test, word2idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn)

In [ ]:
lengths = [len(tokens) for tokens in tokenized_texts]
print(f"Средняя длина: {sum(lengths) / len(lengths):.1f}")
print(f"Медиана: {sorted(lengths)[len(lengths)//2]}")
print(f"99-й перцентиль: {sorted(lengths)[int(len(lengths)*0.99)]}")

Средняя длина: 16.8
Медиана: 16
99-й перцентиль: 35


## Этап 2. Объявление модели LSTM

In [ ]:
device = (
        "cuda" if torch.cuda.is_available() else
        "mps" if torch.backends.mps.is_available() else
        "cpu"
    )
print(f"Using device: {device}")

print("torch.backends.mps.is_available:", torch.backends.mps.is_available())
print("torch.backends.mps.is_built:", torch.backends.mps.is_built())

Using device: cuda
torch.backends.mps.is_available: False
torch.backends.mps.is_built: False


In [ ]:
model = LSTMLanguageModel(vocab_size, word2idx, idx2word).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=word2idx[PAD_TOKEN])
optimizer = optim.Adam(model.parameters(), lr=0.0005)

## Этап 3. Тренировка модели LSTM

In [ ]:
train_losses, val_losses = train_model(model, train_loader, val_loader, criterion, optimizer, idx2word, word2idx)

Epoch 1/10 [Train]:   0%|          | 2/2496 [00:01<18:41,  2.22it/s, loss=9.9000]


[Epoch 1, Batch 0]
  Allocated: 1.47 GB
  Reserved: 5.89 GB


Epoch 1/10 [Train]:   0%|          | 4/2496 [00:01<15:40,  2.65it/s, loss=9.8735]


KeyboardInterrupt: 

In [ ]:
save_model_weight(model)

### Этап 4. Использование предобученного трансформера

In [ ]:
evaluate_transformer(val_loader, idx2word, word2idx, device=device)

In [ ]:
# Если есть явный фаворит — замерено финальное качество на тестовом датасете и отдельно приведены примеры его предсказаний.
#test_model(model, test_loader, idx2word, word2idx, device, criterion)